# CFPB Consumer Complaints: Data Loading and EDA

This notebook starts the project with the official CFPB consumer complaints data.
It focuses on loading the raw dataset, checking the schema, and identifying the first modeling target.

In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path('../data/processed')

# Load the sampled data (created by src/sample_cfpb_data.py)
sample_file = data_dir / 'complaints_sample.csv'

if not sample_file.exists():
    raise FileNotFoundError(
        f'{sample_file} not found. Run: python src/sample_cfpb_data.py --sample-size 5000'
    )

df = pd.read_csv(sample_file)
print(f"Loaded {len(df):,} rows from {sample_file.name}")
print(f"Shape: {df.shape}")
df.head(2)

Loaded 5,000 rows from complaints_sample.csv
Shape: (5000, 18)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2026-02-24,Debt collection,I do not know,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,NaN,NaN,"Affirm Holdings, Inc",CA,91202,NaN,NaN,Web,2026-02-24,Closed with explanation,Yes,NaN,19737724
1,2025-11-05,Prepaid card,General-purpose prepaid card,Problem getting a card or closing an account,Trouble closing card,On XXXX I received a prepaid debit card from N...,Company has responded to the consumer and the ...,Netspend Corporation,NC,28078,NaN,Consent provided,Web,2025-11-05,Closed with explanation,Yes,NaN,17039807


In [3]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
})
summary.sort_values('missing_pct', ascending=False).head(20)

,dtype,missing_count,missing_pct
Tags,str,4759,95.18
Consumer disputed?,str,4754,95.08
Consumer complaint narrative,str,3722,74.44
Company public response,str,2307,46.14
Consumer consent provided?,str,759,15.18
Sub-issue,str,298,5.96
Sub-product,str,72,1.44
State,str,16,0.32
ZIP code,str,8,0.16
Date received,str,0,0.00


In [4]:
target_col = 'Timely response?'
if target_col in df.columns:
    display(df[target_col].value_counts(dropna=False))
else:
    print(f'{target_col} not found. Available columns:')
    print(list(df.columns[:50]))

Timely response?
Yes    4959
No       41
Name: count, dtype: int64

In [5]:
narrative_col = 'Consumer complaint narrative'
if narrative_col in df.columns:
    narrative_availability = df[narrative_col].notna().mean() * 100
    print(f'Narrative availability: {narrative_availability:.2f}%')
else:
    print(f'{narrative_col} not found.')

Narrative availability: 25.56%
